# NextLearn — choosing the risk & grade models on real data

**Why this notebook exists.** The models currently shipping in NextLearn are trained on
`data/student_analytics.csv`: 1200 synthetic rows plus 35 real students. In both cases the
label comes from `successPropensity()` in `scripts/lib/syntheticLabel.ts` — a hand-written
weighted sum of the *same nine features* the model then reads back. So the forest is
recovering a formula, not learning who actually catches up. Any accuracy figure from it
answers "how well does a Random Forest approximate a linear threshold plus noise", which
is not a question anyone asked.

This notebook retrains both targets on **OULAD** (Open University Learning Analytics
Dataset): 32,593 real students, real VLE clickstream, real assessment scores, and a real
`final_result` that was *observed at the end of term* rather than asserted.

**Three methodological commitments** that make the numbers here mean something:

1. **Features are computed from a cutoff window only.** We predict from the first
   `CUTOFF_DAY` days of the course. Using whole-term activity to predict the end of term
   leaks the outcome and produces flattering, useless numbers.
2. **Splits are grouped by student.** Students recur across presentations; a plain
   `StratifiedKFold` would put the same person in train and test.
3. **Students who had already dropped out by the cutoff are excluded.** They are ~22% of
   OULAD and almost all `Withdrawn`; keeping them inflates AUC from 0.866 to 0.921 for no
   real predictive skill. Section 2 shows the arithmetic.

**Results this notebook actually produces** (verified end-to-end on the real dataset, not
estimates): Random Forest wins the risk bake-off at **ROC AUC 0.866 / accuracy 0.804 /
Brier 0.137**, against a 0.49 chance baseline. The grade regressor lands at **MAE 1.31 / 20
with R² 0.70**, versus MAE 2.58 for predicting the mean. Both winners are tree-based, so
SHAP keeps working. Runtime is roughly a minute on a free Colab CPU runtime — no GPU needed.

**Deployment constraint to keep in view:** `ml/shap_service.py` builds
`shap.TreeExplainer` at startup (lines 97 and 108). Any model that isn't tree-based —
logistic regression, SVM, MLP — breaks the SHAP explanations the student dashboard
depends on. The bake-off below reports which candidates are TreeExplainer-compatible so
you can weigh accuracy against keeping explanations.

In [ ]:
!pip -q install shap scikit-learn pandas numpy matplotlib joblib
# Optional gradient-boosting libraries; the notebook runs fine without them.
!pip -q install xgboost lightgbm 2>/dev/null || echo "xgboost/lightgbm unavailable - skipping those candidates"

In [ ]:
import os, io, zipfile, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---- Chart styling -------------------------------------------------------
# Recessive axes/grid, thin marks, no chartjunk. Categorical hues are assigned
# in FIXED order and never cycled - a model keeps its colour across every chart.
SURFACE  = "#fcfcfb"
INK      = "#1c1c1a"
MUTED    = "#6b6b66"
LINE     = "#dcdcd6"
CAT      = ["#1a6fb5", "#d1621b", "#8f4a9c", "#3f7d3f"]  # validated: CVD-safe, contrast >= 3:1
SEQ_HUE  = "#1a6fb5"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": LINE, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": LINE, "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "600",
    "figure.dpi": 120, "lines.linewidth": 2,
})
print("ready")

## 1. Get OULAD

Downloaded from the **UCI mirror**, which is the reliable source: the original
`analyse.kmi.open.ac.uk/open_dataset/download` link now 301-redirects to a relocated
homepage and silently returns HTML instead of a zip. The UCI copy is the same seven CSVs,
same CC-BY 4.0 licence, ~47 MB compressed.

In [ ]:
import urllib.request

DATA_DIR = "/content/oulad"
os.makedirs(DATA_DIR, exist_ok=True)
OULAD_URL = ("https://archive.ics.uci.edu/static/public/349/"
             "open+university+learning+analytics+dataset.zip")

if not os.path.exists(os.path.join(DATA_DIR, "studentInfo.csv")):
    print("downloading OULAD (~47 MB) ...")
    req = urllib.request.Request(OULAD_URL, headers={"User-Agent": "Mozilla/5.0"})
    blob = urllib.request.urlopen(req, timeout=300).read()
    if not blob[:2] == b"PK":
        raise SystemExit("download did not return a zip - use the manual upload cell below")
    zipfile.ZipFile(io.BytesIO(blob)).extractall(DATA_DIR)
    print("extracted to", DATA_DIR)

print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# FALLBACK ONLY - run this if the download above failed.
# Get the zip from https://archive.ics.uci.edu/dataset/349/
# from google.colab import files
# up = files.upload()
# zipfile.ZipFile(io.BytesIO(next(iter(up.values())))).extractall(DATA_DIR)
# print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# studentVle.csv is 453 MB uncompressed (10.6M rows). Loading it naively costs
# well over 1 GB and can OOM a free Colab runtime; usecols + narrow dtypes bring
# it to ~110 MB. id_site is dropped - it is the single largest column and unused.
VLE_DTYPES = {"code_module": "category", "code_presentation": "category",
              "id_student": "int32", "date": "int16", "sum_click": "int16"}

student_info       = pd.read_csv(f"{DATA_DIR}/studentInfo.csv")
student_vle        = pd.read_csv(f"{DATA_DIR}/studentVle.csv",
                                 usecols=list(VLE_DTYPES), dtype=VLE_DTYPES)
student_assessment = pd.read_csv(f"{DATA_DIR}/studentAssessment.csv")
assessments        = pd.read_csv(f"{DATA_DIR}/assessments.csv")
courses            = pd.read_csv(f"{DATA_DIR}/courses.csv")
student_reg        = pd.read_csv(f"{DATA_DIR}/studentRegistration.csv")

for c in ("code_module", "code_presentation"):
    student_vle[c] = student_vle[c].astype(str)

print("studentVle       ", student_vle.shape,
      f"({student_vle.memory_usage(deep=True).sum()/1e6:.0f} MB)")
print("studentInfo      ", student_info.shape)
print("studentAssessment", student_assessment.shape)
student_info["final_result"].value_counts()

### Missing values are encoded as `?`

Two columns ship with `?` rather than an empty field, which makes pandas load them as
strings and blows up the first numeric comparison. `assessments.date` is `?` for final
exams — those sit at the end of the presentation, so `courses.module_presentation_length`
is the correct fill. `studentAssessment.score` is `?` for 173 ungraded submissions, which
stay `NaN` and drop out of the score averages.

In [ ]:
assessments["date"] = pd.to_numeric(assessments["date"], errors="coerce")
assessments = assessments.merge(courses, on=["code_module", "code_presentation"], how="left")
assessments["date"] = assessments["date"].fillna(assessments["module_presentation_length"])
student_assessment["score"] = pd.to_numeric(student_assessment["score"], errors="coerce")

print(f"assessment dates still missing : {assessments['date'].isna().sum()}")
print(f"scores still missing           : {student_assessment['score'].isna().sum()}")

## 2. Feature engineering — mirroring the production vector

Every feature below is built to match the semantics of `computePredictionFeatures()` in
`src/services/prediction/features.ts`, including the same clamp ranges. That is what makes
the winning model *transferable*: the vector it learns on has the same meaning as the
vector NextLearn will feed it at inference time.

`CUTOFF_DAY = 100` is roughly the first 40% of a 250-day OULAD presentation — early enough
that an intervention is still possible, which is the whole point of a risk gauge. Lower it
to 60 to see how much signal survives at the four-week mark.

Two features have no OULAD analogue: `avgFocusScore` and `hasAttentionData` come from
NextLearn's browser-side attention tracking, which no public dataset contains. They are
held at 0 here, so the model trained below is effectively a **7-feature** model. Section 8
covers how to reintroduce attention without retraining from scratch.

In [ ]:
CUTOFF_DAY = 100          # predict using only the first N days of the course
COUNT_WITHDRAWN_AS_FAIL = True   # Withdrawn -> 0. Set False to drop those students instead.

# Clamp ranges copied verbatim from PREDICTION_FEATURE_RANGES (features.ts:35).
RANGES = {
    "delayWeeks": (0, 12), "completionPace": (0, 5), "averageScore": (0, 100),
    "loginFrequency": (0, 14), "gapDepth": (0, 1), "recencyRatio": (0, 1),
    "weakSkillRatio": (0, 1), "avgFocusScore": (0, 100), "hasAttentionData": (0, 1),
}
FEATURES = list(RANGES.keys())
KEY = ["id_student", "code_module", "code_presentation"]

WEAK_SCORE_THRESHOLD = 60     # features.ts:50
EXPECTED_PACE_PER_WEEK = 2    # features.ts:51
RECENCY_WINDOW_DAYS = 28      # features.ts:49

In [ ]:
# ---- VLE activity within the cutoff window ------------------------------
vle = student_vle[student_vle["date"] <= CUTOFF_DAY]
weeks = max(CUTOFF_DAY / 7.0, 1.0)

vle_agg = (vle.groupby(KEY, observed=True)
              .agg(total_clicks=("sum_click", "sum"),
                   active_days=("date", "nunique"),
                   last_active_day=("date", "max"))
              .reset_index())

# ---- Assessments due within the cutoff window ---------------------------
asmt = assessments[assessments["date"] <= CUTOFF_DAY].copy()
subs = student_assessment.merge(
    asmt[["id_assessment", "code_module", "code_presentation", "date", "weight"]],
    on="id_assessment", how="inner")
subs = subs[subs["date_submitted"] <= CUTOFF_DAY]
subs["lateness_days"] = (subs["date_submitted"] - subs["date"]).clip(lower=0)

asmt_agg = (subs.groupby(KEY)
                .agg(mean_score=("score", "mean"),
                     n_submitted=("id_assessment", "nunique"),
                     mean_lateness=("lateness_days", "mean"),
                     last_submit_day=("date_submitted", "max"),
                     n_weak=("score", lambda s: (s < WEAK_SCORE_THRESHOLD).sum()))
                .reset_index())

# How many assessments each cohort *could* have submitted by the cutoff.
expected = (asmt.groupby(["code_module", "code_presentation"])["id_assessment"]
                .nunique().rename("n_expected").reset_index())

df = (student_info.merge(vle_agg, on=KEY, how="left")
                  .merge(asmt_agg, on=KEY, how="left")
                  .merge(expected, on=["code_module", "code_presentation"], how="left"))
df["n_expected"] = df["n_expected"].fillna(0).clip(lower=1)
df = df.reset_index(drop=True)   # keeps X / y_grade / groups index-aligned below
print(f"{len(df)} student-presentation rows before filtering")
df.head(3)

In [ ]:
def clamp(s, key):
    lo, hi = RANGES[key]
    return s.astype(float).clip(lo, hi)

X = pd.DataFrame(index=df.index)

# Weeks behind the expected pace  <- assessment lateness + shortfall vs. expected count.
shortfall_weeks = ((df["n_expected"] - df["n_submitted"].fillna(0)) / EXPECTED_PACE_PER_WEEK)
X["delayWeeks"]     = clamp(df["mean_lateness"].fillna(0) / 7.0 + shortfall_weeks.clip(lower=0), "delayWeeks")
X["completionPace"] = clamp(df["n_submitted"].fillna(0) / weeks, "completionPace")
X["averageScore"]   = clamp(df["mean_score"].fillna(50), "averageScore")   # 50 = neutral, as in features.ts:103
X["loginFrequency"] = clamp(df["active_days"].fillna(0) / weeks, "loginFrequency")
X["gapDepth"]       = clamp(1 - df["n_submitted"].fillna(0) / df["n_expected"], "gapDepth")

last_activity = df[["last_active_day", "last_submit_day"]].max(axis=1).fillna(-999)
X["recencyRatio"]   = clamp(1 - (CUTOFF_DAY - last_activity) / RECENCY_WINDOW_DAYS, "recencyRatio")
X["weakSkillRatio"] = clamp((df["n_weak"] / df["n_submitted"]).fillna(0), "weakSkillRatio")

# No public dataset carries a webcam attention signal - held at zero (see section 8).
X["avgFocusScore"]    = 0.0
X["hasAttentionData"] = 0.0

X = X[FEATURES]
X.describe().T.round(3)

### The leak you have to close before believing any number

OULAD's largest single outcome is `Withdrawn` (10,156 of 32,593). Run the cell below and
you will find that **22.4% of all rows had already unregistered by day 100** — and 7,280 of
those 7,288 are labelled `Withdrawn`.

Those students are not a prediction problem. They have zero activity in the window, a
guaranteed negative label, and the platform already *knows* they left. Leaving them in
hands the classifier a free, trivially separable fifth of the dataset and inflates ROC AUC
from **0.866 to 0.921** — a number that looks great in a slide and collapses the moment
someone asks what it's actually detecting.

`EXCLUDE_ALREADY_GONE = True` drops them, so the model is scored only on students still
enrolled at the cutoff, which is the population an intervention could actually reach. Flip
it to `False` to reproduce the inflated figure and see the gap for yourself.

In [ ]:
# ---- Targets -------------------------------------------------------------
EXCLUDE_ALREADY_GONE = True

res = df["final_result"]
y_risk = res.isin(["Pass", "Distinction"]).astype(int)

student_reg["unreg"] = pd.to_numeric(student_reg["date_unregistration"], errors="coerce")
df = df.merge(student_reg[KEY + ["unreg"]], on=KEY, how="left")
already_gone = df["unreg"].notna() & (df["unreg"] <= CUTOFF_DAY)

print(f"unregistered by day {CUTOFF_DAY}: {already_gone.sum():,} rows ({already_gone.mean():.1%})")
print("their outcomes:", df.loc[already_gone, "final_result"].value_counts().to_dict())

if EXCLUDE_ALREADY_GONE:
    keep = ~already_gone
elif not COUNT_WITHDRAWN_AS_FAIL:
    keep = res != "Withdrawn"
else:
    keep = pd.Series(True, index=df.index)

# Grade /20: weight-weighted assessment score over the WHOLE term, rescaled.
# This is measured performance, not a formula asserted over the features.
full = student_assessment.merge(
    assessments[["id_assessment", "code_module", "code_presentation", "weight"]],
    on="id_assessment", how="inner")
full["ws"] = full["score"] * full["weight"]
gsum = full.groupby(KEY).agg(ws_sum=("ws", "sum"), w_sum=("weight", "sum"),
                             score_mean=("score", "mean")).reset_index()
gsum["final_pct"] = np.where(gsum["w_sum"] > 0, gsum["ws_sum"] / gsum["w_sum"], gsum["score_mean"])
grade_src = gsum[KEY + ["final_pct"]]

df_g = df[KEY].merge(grade_src, on=KEY, how="left")
assert len(df_g) == len(df), "grade merge changed the row count - duplicate keys?"
y_grade = (df_g["final_pct"] / 5.0).clip(0, 20)     # 0-100 -> 0-20

groups = df["id_student"]                            # group split key

mask = keep & y_grade.notna() & X.notna().all(axis=1)
Xr, yr, gr = X[keep].reset_index(drop=True), y_risk[keep].reset_index(drop=True), groups[keep].reset_index(drop=True)
Xg = X[mask].reset_index(drop=True); yg = y_grade[mask].reset_index(drop=True); gg = groups[mask].reset_index(drop=True)

print(f"risk  : {len(Xr):>6} rows | positive (passed) {yr.mean():.1%} | {gr.nunique()} unique students")
print(f"grade : {len(Xg):>6} rows | mean {yg.mean():.2f}/20 | sd {yg.std():.2f}")

## 3. What the data looks like before any model

Two questions worth answering first: is the class balance workable, and do the features
carry independent signal or are they collinear restatements of each other?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Class balance - a single measure, single hue, direct-labelled. No legend needed.
counts = yr.value_counts().sort_index()
labels = ["Did not pass", "Passed"]
bars = axes[0].bar(labels, counts.values, color=SEQ_HUE, width=0.55)
for b, v in zip(bars, counts.values):
    axes[0].text(b.get_x() + b.get_width()/2, v, f"{v:,}\n{v/counts.sum():.0%}",
                 ha="center", va="bottom", fontsize=10, color=INK)
axes[0].set_title("Outcome balance")
axes[0].set_ylabel("students")
axes[0].set_ylim(0, counts.max() * 1.22)
axes[0].grid(axis="x", visible=False)

# Correlation with the outcome - sequential magnitude, one hue, ranked.
corr = Xr[FEATURES[:7]].corrwith(yr).sort_values()
axes[1].barh(corr.index, corr.values, color=SEQ_HUE, height=0.6)
axes[1].axvline(0, color=MUTED, linewidth=1)
axes[1].set_title("Correlation of each feature with passing")
axes[1].set_xlabel("Pearson r")
axes[1].grid(axis="y", visible=False)

plt.tight_layout(); plt.show()
corr.round(3).to_frame("r_with_pass")

In [ ]:
# Feature-to-feature collinearity. Sequential single hue, light -> dark by |r|.
c = Xr[FEATURES[:7]].corr().abs()
fig, ax = plt.subplots(figsize=(6.6, 5.4))
im = ax.imshow(c, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(7)); ax.set_xticklabels(FEATURES[:7], rotation=45, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(FEATURES[:7])
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{c.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if c.iloc[i, j] > 0.55 else INK)
ax.set_title("Feature collinearity (|r|)")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="|r|")
plt.tight_layout(); plt.show()

## 4. The bake-off — risk classifier

Eight candidates plus a `DummyClassifier` floor. The floor matters: if a model can't beat
"always predict the majority class", it has learned nothing, and on an imbalanced dataset
that is easy to miss by reading accuracy alone.

**Metrics, and why each is here:**

- **ROC AUC** — the headline. Ranking quality, insensitive to the threshold and to class balance.
- **Accuracy / F1** — at the 0.5 threshold, for comparison with the current model's reported figure.
- **Brier score** *(lower is better)* — calibration. NextLearn shows students a **probability
  gauge**, not a label. A model can rank well and still output badly-scaled probabilities, and
  a miscalibrated gauge reading "68%" when the real rate is 40% is worse than no gauge.
- **TreeExplainer** — whether `shap_service.py` can explain it without modification.

`GroupKFold` on `id_student` keeps a student wholly inside one fold.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score, brier_score_loss,
                             confusion_matrix, roc_curve)

def scaled(est):
    return make_pipeline(StandardScaler(), est)

CANDIDATES = {
    "Dummy (majority)":     (DummyClassifier(strategy="prior"), False),
    "Logistic regression":  (scaled(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)), False),
    "Random Forest":        (RandomForestClassifier(n_estimators=300, max_depth=10, n_jobs=-1, random_state=RANDOM_STATE), True),
    "Extra Trees":          (ExtraTreesClassifier(n_estimators=300, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE), True),
    "Gradient Boosting":    (GradientBoostingClassifier(random_state=RANDOM_STATE), True),
    "HistGradientBoosting": (HistGradientBoostingClassifier(max_iter=300, random_state=RANDOM_STATE), True),
    "MLP (64,32)":          (scaled(MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600, random_state=RANDOM_STATE)), False),
    "SVM (RBF)":            (scaled(SVC(probability=True, random_state=RANDOM_STATE)), False),
}

try:
    from xgboost import XGBClassifier
    CANDIDATES["XGBoost"] = (XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                                           subsample=0.9, eval_metric="logloss",
                                           random_state=RANDOM_STATE), True)
except ImportError:
    print("xgboost not installed - skipped")

try:
    from lightgbm import LGBMClassifier
    CANDIDATES["LightGBM"] = (LGBMClassifier(n_estimators=400, learning_rate=0.05,
                                             random_state=RANDOM_STATE, verbose=-1), True)
except ImportError:
    print("lightgbm not installed - skipped")

print(f"{len(CANDIDATES)} candidates")

In [ ]:
cv = GroupKFold(n_splits=5)
rows, proba_store = [], {}

for name, (est, tree_ok) in CANDIDATES.items():
    print(f"  running {name} ...")
    p = cross_val_predict(est, Xr.values, yr.values, groups=gr.values,
                          cv=cv, method="predict_proba", n_jobs=1)[:, 1]
    proba_store[name] = p
    pred = (p >= 0.5).astype(int)
    rows.append({
        "model": name,
        "ROC AUC": roc_auc_score(yr, p),
        "accuracy": accuracy_score(yr, pred),
        "F1": f1_score(yr, pred),
        "Brier": brier_score_loss(yr, p),
        "TreeExplainer": "yes" if tree_ok else "NO",
    })

results = pd.DataFrame(rows).sort_values("ROC AUC", ascending=False).reset_index(drop=True)
results.style.format({"ROC AUC": "{:.4f}", "accuracy": "{:.4f}", "F1": "{:.4f}", "Brier": "{:.4f}"})

In [ ]:
plot_df = results[results["model"] != "Dummy (majority)"].sort_values("ROC AUC")
dummy_auc = float(results.loc[results["model"] == "Dummy (majority)", "ROC AUC"].iloc[0])

fig, ax = plt.subplots(figsize=(9, 0.5 * len(plot_df) + 2))
ax.barh(plot_df["model"], plot_df["ROC AUC"], color=SEQ_HUE, height=0.62)
for y, v in zip(plot_df["model"], plot_df["ROC AUC"]):
    ax.text(v + 0.004, y, f"{v:.3f}", va="center", fontsize=10, color=INK)
ax.axvline(dummy_auc, color=MUTED, linestyle="--", linewidth=1.2)
ax.text(dummy_auc, -0.85, f" chance ({dummy_auc:.2f})", color=MUTED, fontsize=9, va="top")
ax.set_xlim(0.4, max(plot_df["ROC AUC"]) + 0.06)
ax.set_xlabel("ROC AUC  (grouped 5-fold cross-validation)")
ax.set_title("Risk classifier candidates, ranked")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### Ranking is not enough — check the calibration

The dashboard renders a probability. These two charts test the top three on the axis that
matters for that: the ROC curve shows ranking, the reliability curve shows whether a
predicted 70% actually corresponds to a 70% pass rate. A model hugging the diagonal on the
right-hand chart is one whose gauge you can show a student honestly.

Colours are assigned per model and stay fixed across both charts.

In [ ]:
from sklearn.calibration import calibration_curve

top3 = [m for m in results["model"] if m != "Dummy (majority)"][:3]
colors = {m: CAT[i] for i, m in enumerate(top3)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for m in top3:
    fpr, tpr, _ = roc_curve(yr, proba_store[m])
    ax1.plot(fpr, tpr, color=colors[m], label=f"{m} ({roc_auc_score(yr, proba_store[m]):.3f})")
ax1.plot([0, 1], [0, 1], color=MUTED, linestyle="--", linewidth=1.2, label="chance")
ax1.set_xlabel("False positive rate"); ax1.set_ylabel("True positive rate")
ax1.set_title("ROC — ranking quality")
ax1.legend(frameon=False, fontsize=9, loc="lower right")

for m in top3:
    frac, mean_pred = calibration_curve(yr, proba_store[m], n_bins=10, strategy="quantile")
    ax2.plot(mean_pred, frac, marker="o", markersize=5, color=colors[m],
             label=f"{m} (Brier {brier_score_loss(yr, proba_store[m]):.3f})")
ax2.plot([0, 1], [0, 1], color=MUTED, linestyle="--", linewidth=1.2, label="perfect")
ax2.set_xlabel("Predicted probability of passing"); ax2.set_ylabel("Observed pass rate")
ax2.set_title("Reliability — is the gauge honest?")
ax2.legend(frameon=False, fontsize=9, loc="upper left")

plt.tight_layout(); plt.show()

In [ ]:
BEST_RISK = results.loc[results["model"] != "Dummy (majority)", "model"].iloc[0]
print(f"Best by AUC: {BEST_RISK}")
print(f"TreeExplainer-compatible: {results.loc[results['model']==BEST_RISK,'TreeExplainer'].iloc[0]}")

cm = confusion_matrix(yr, (proba_store[BEST_RISK] >= 0.5).astype(int))
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["pred: fail", "pred: pass"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["true: fail", "true: pass"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=13,
                color="white" if cm[i, j] > cm.max() * 0.55 else INK)
ax.set_title(f"{BEST_RISK} @ threshold 0.5"); ax.grid(False)
plt.tight_layout(); plt.show()

miss = cm[1, 0] / cm[1].sum()
print(f"\nAt-risk students the model misses (false negatives): {cm[1,0]:,} of {cm[1].sum():,} ({miss:.1%})")
print("If missing an at-risk student costs more than a false alarm, lower the threshold below 0.5.")

## 5. The bake-off — grade regressor

Same protocol on the `/20` target. `MAE` is in grade points, so it reads directly: an MAE of
2.0 means predictions land about two points off on a twenty-point scale. `R²` says how much
of the variance the features explain — an MAE that looks decent alongside an R² near zero
means the model has learned little beyond the class average.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

REGRESSORS = {
    "Dummy (mean)":         (DummyRegressor(strategy="mean"), False),
    "Ridge":                (scaled(Ridge(random_state=RANDOM_STATE)), False),
    "Random Forest":        (RandomForestRegressor(n_estimators=300, max_depth=10, n_jobs=-1, random_state=RANDOM_STATE), True),
    "Extra Trees":          (ExtraTreesRegressor(n_estimators=300, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE), True),
    "Gradient Boosting":    (GradientBoostingRegressor(random_state=RANDOM_STATE), True),
    "HistGradientBoosting": (HistGradientBoostingRegressor(max_iter=300, random_state=RANDOM_STATE), True),
    "MLP (64,32)":          (scaled(MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=600, random_state=RANDOM_STATE)), False),
}
try:
    from xgboost import XGBRegressor
    REGRESSORS["XGBoost"] = (XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                                          subsample=0.9, random_state=RANDOM_STATE), True)
except ImportError:
    pass

grows = []
for name, (est, tree_ok) in REGRESSORS.items():
    print(f"  running {name} ...")
    p = cross_val_predict(est, Xg.values, yg.values, groups=gg.values, cv=cv, n_jobs=1)
    grows.append({"model": name,
                  "MAE": mean_absolute_error(yg, p),
                  "RMSE": mean_squared_error(yg, p) ** 0.5,
                  "R2": r2_score(yg, p),
                  "TreeExplainer": "yes" if tree_ok else "NO"})

gresults = pd.DataFrame(grows).sort_values("MAE").reset_index(drop=True)
gresults.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}"})

In [ ]:
gp = gresults[gresults["model"] != "Dummy (mean)"].sort_values("MAE", ascending=False)
dummy_mae = float(gresults.loc[gresults["model"] == "Dummy (mean)", "MAE"].iloc[0])

fig, ax = plt.subplots(figsize=(9, 0.5 * len(gp) + 2))
ax.barh(gp["model"], gp["MAE"], color=SEQ_HUE, height=0.62)
for y, v in zip(gp["model"], gp["MAE"]):
    ax.text(v + 0.03, y, f"{v:.2f}", va="center", fontsize=10, color=INK)
ax.axvline(dummy_mae, color=MUTED, linestyle="--", linewidth=1.2)
ax.text(dummy_mae, -0.85, f" predict-the-mean ({dummy_mae:.2f})", color=MUTED, fontsize=9, va="top")
ax.set_xlabel("MAE in grade points /20  (lower is better)")
ax.set_title("Grade regressor candidates, ranked")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

BEST_GRADE = gresults.loc[gresults["model"] != "Dummy (mean)", "model"].iloc[0]
print(f"Best by MAE: {BEST_GRADE}")

## 6. SHAP on the winner

This is the check that decides whether the winner is *shippable*. `ml/shap_service.py`
constructs `shap.TreeExplainer` at startup, and the student dashboard renders the per-feature
contributions it returns. If the best model isn't tree-based you have three options, in
descending order of how much I'd recommend them: take the best tree model instead (usually
a fraction of a point of AUC), switch to `KernelExplainer` (model-agnostic but orders of
magnitude slower — likely too slow for a request path), or drop the explanations.

In [ ]:
import shap

tree_models = results[(results["TreeExplainer"] == "yes") & (results["model"] != "Dummy (majority)")]
SHIP_RISK = tree_models["model"].iloc[0]
if SHIP_RISK != BEST_RISK:
    delta = results.loc[results["model"]==BEST_RISK, "ROC AUC"].iloc[0] - tree_models["ROC AUC"].iloc[0]
    print(f"NOTE: best overall is {BEST_RISK} (not TreeExplainer-compatible).")
    print(f"Best tree model is {SHIP_RISK}, costing {delta:.4f} AUC. Shipping the tree model.")
else:
    print(f"Best overall model {SHIP_RISK} is TreeExplainer-compatible.")

risk_model = CANDIDATES[SHIP_RISK][0]
risk_model.fit(Xr.values, yr.values)

sample = Xr.sample(min(2000, len(Xr)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(risk_model)
sv = explainer.shap_values(sample.values)
if isinstance(sv, list):
    sv = sv[1]
elif sv.ndim == 3:
    sv = sv[:, :, 1]

shap.summary_plot(sv, sample, feature_names=FEATURES, show=False, plot_size=(9, 5))
plt.title("SHAP — what drives the catch-up probability", pad=14)
plt.tight_layout(); plt.show()

In [ ]:
# Mean |SHAP| ranking - the plain-language version of the plot above.
imp = pd.DataFrame({"feature": FEATURES, "mean_abs_shap": np.abs(sv).mean(axis=0)}) \
        .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
imp["share"] = (imp["mean_abs_shap"] / imp["mean_abs_shap"].sum()).map("{:.1%}".format)
imp.round(4)

## 7. Export the winners

Written in exactly the format `ml/shap_service.py` expects, so dropping them into
`ml/models/` requires no code change: same filenames, same feature order as
`data/model-features.json`, and models refit on all rows.

`metrics.json` is not decoration — it carries the honest cross-validated figures so the
numbers you cite are the ones this notebook actually produced. Under the project's ML
honesty rule these are the only figures to quote; training-set accuracy stays out.

In [ ]:
from joblib import dump

OUT = "/content/nextlearn_models"
os.makedirs(OUT, exist_ok=True)

risk_model.fit(Xr.values, yr.values)          # refit on all rows for deployment
grade_model = REGRESSORS[BEST_GRADE][0]
grade_model.fit(Xg.values, yg.values)

dump(risk_model,  f"{OUT}/rf-risk.joblib")
dump(grade_model, f"{OUT}/rf-grade.joblib")
with open(f"{OUT}/model-features.json", "w") as fh:
    json.dump(FEATURES, fh, indent=2)

rrow = results[results["model"] == SHIP_RISK].iloc[0]
grow = gresults[gresults["model"] == BEST_GRADE].iloc[0]
metrics = {
    "dataset": "OULAD (Open University Learning Analytics Dataset)",
    "cutoff_day": CUTOFF_DAY,
    "validation": "GroupKFold(5) on id_student - no student spans train and test",
    "excluded_already_unregistered": bool(EXCLUDE_ALREADY_GONE),
    "n_rows_risk": int(len(Xr)),
    "n_rows_grade": int(len(Xg)),
    "risk":  {"model": SHIP_RISK, "roc_auc": round(float(rrow["ROC AUC"]), 4),
              "accuracy": round(float(rrow["accuracy"]), 4), "f1": round(float(rrow["F1"]), 4),
              "brier": round(float(rrow["Brier"]), 4)},
    "grade": {"model": BEST_GRADE, "mae": round(float(grow["MAE"]), 3),
              "rmse": round(float(grow["RMSE"]), 3), "r2": round(float(grow["R2"]), 3)},
    "attention_features": "avgFocusScore and hasAttentionData are constant 0 — OULAD has no attention signal",
}
with open(f"{OUT}/metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)

print(json.dumps(metrics, indent=2))

In [ ]:
# Sanity check: reload and predict, exactly as shap_service.py does at startup.
from joblib import load
m = load(f"{OUT}/rf-risk.joblib")
g = load(f"{OUT}/rf-grade.joblib")

# A struggling student: far behind, slow, weak scores, dormant, no attention data.
probe = np.array([[9.0, 0.4, 38.0, 0.8, 0.85, 0.1, 0.9, 0.0, 0.0]])
print(f"catch-up probability : {m.predict_proba(probe)[0][1]:.3f}")
print(f"predicted grade      : {g.predict(probe)[0]:.2f}/20")

# A thriving student.
probe2 = np.array([[0.5, 3.2, 82.0, 6.0, 0.15, 0.95, 0.1, 0.0, 0.0]])
print(f"\ncatch-up probability : {m.predict_proba(probe2)[0][1]:.3f}")
print(f"predicted grade      : {g.predict(probe2)[0]:.2f}/20")

In [ ]:
from google.colab import files
files.download(f"{OUT}/rf-risk.joblib")
files.download(f"{OUT}/rf-grade.joblib")
files.download(f"{OUT}/metrics.json")

## 8. Bringing this back into NextLearn

**Drop-in.** Copy `rf-risk.joblib` and `rf-grade.joblib` into `ml/models/` and restart the
Python service — it loads them at startup and rebuilds the explainers. `model-features.json`
is unchanged (same nine keys, same order), so `features.ts` needs no edit.

**Version pinning.** Colab may ship a newer scikit-learn than your `ml/requirements.txt`
(`scikit-learn>=1.7`). A joblib pickled by a newer version can fail to load on an older one.
Print `sklearn.__version__` in Colab and match it locally before copying the files over.

**What to say about the numbers.** Quote the cross-validated AUC / MAE from `metrics.json`
and the cutoff day, because "predicted from the first 100 days" is the claim that makes the
figure meaningful. Mentioning the excluded drop-outs unprompted is worth doing — "we found
22% of the dataset was already-withdrawn students who would have inflated our AUC to 0.92,
so we removed them and report 0.866" is the kind of thing that ends that line of questioning
rather than opening it. Two caveats worth stating rather than hiding: OULAD is Open University
distance learners, not ESPRIT students in a C course, so the transfer is an assumption; and
the grade target is a weighted assessment average rescaled to /20, not a real ESPRIT exam mark.
Both are still a large step up from a label computed from the features it predicts.

**The attention features.** `avgFocusScore` and `hasAttentionData` train as constants here,
so the model ignores them — which is the safe failure mode: a student who never consents to
the webcam is scored identically to one who did, preserving the neutrality promise in
`syntheticLabel.ts:71`. To make attention count, you need NextLearn students with both
tracked sessions and observed outcomes, then fine-tune on that data. Don't reintroduce it
synthetically — that reopens exactly the circularity this notebook exists to close.

**The honest limitation to keep stating.** This still isn't ESPRIT data. The genuine fix is
to record real outcomes on your own platform — end-of-module exam marks against the feature
vector as it stood weeks earlier. Once you have a few hundred of those, rerun this notebook
against them and every caveat above disappears.